# Reliability, Monitoring & Cost Optimization

Topics:
- **`with_retry` decorator** — exponential backoff with jitter
- **`CircuitBreaker`** — three states: closed → open → half-open
- **`FallbackChain`** — try multiple models in priority order
- **Robust LangGraph agent** — retry logic baked into graph routing
- **`JSONFormatter`** — structured logs for log aggregation systems
- **`MetricsCollector`** — latency, token, error, and cache-hit tracking
- **`ModelRouter`** — classify complexity, route cheap vs. expensive model
- **`SemanticCache` / `CachedLLM`** — skip LLM calls for seen queries
- **`TokenBudget`** — per-request token ceiling

```
Request flow:
input → [retry/circuit breaker] → LLM (with fallbacks)
                                      ↓ instrumented
                                   response ← cache hit?
                                      ↓
                                   metrics collected, JSON logged
```

In [ ]:
import time, random, hashlib, json
from typing import Literal, Optional, Callable
from functools import wraps
import logging
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
import operator
from langsmith import traceable

## 1. Retry with Exponential Backoff

The `@with_retry` decorator wraps any function and retries up to `max_retries` times on failure. Delay grows exponentially (`base_delay * 2^attempt`) and is capped at `max_delay`. **Jitter** (`* (0.5 + random())`) prevents all retrying clients from hitting the server at exactly the same moment (thundering herd).

```python
delay = min(base_delay * (2 ** attempt), max_delay)
delay = delay * (0.5 + random.random())  # jitter
```

In [ ]:
def with_retry(
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    exceptions: tuple = (Exception,),
):
    """Retry decorator with exponential backoff and jitter."""
    def decorator(func: Callable):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    if attempt < max_retries - 1:
                        delay = min(base_delay * (2 ** attempt), max_delay)
                        delay = delay * (0.5 + random.random())  # jitter
                        print(f"Attempt {attempt + 1} failed: {e}. Retrying in {delay:.1f}s...")
                        time.sleep(delay)
            raise last_exception
        return wrapper
    return decorator


@with_retry(max_retries=3, base_delay=0.2)  # short delay for demo
def unreliable_api_call(query: str) -> str:
    if random.random() < 0.5:
        raise ConnectionError("Simulated API failure")
    return f"Success: {query}"


for i in range(3):
    try:
        result = unreliable_api_call(f"Query {i}")
        print(f"OK: {result}")
    except Exception as e:
        print(f"Failed after retries: {e}")

## 2. Circuit Breaker

A circuit breaker has three states:

| State | Behavior |
|-------|----------|
| **closed** | Normal — calls pass through |
| **open** | Failures exceeded threshold — calls immediately rejected (fast-fail) |
| **half-open** | Recovery window passed — one trial call allowed; success → closed, failure → open |

Fast-fail when the service is down protects both the caller (no unnecessary waits) and the downstream service (no pile-up of timed-out requests).

In [ ]:
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 5, recovery_timeout: float = 30.0):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failures = 0
        self.last_failure_time = 0
        self.state = "closed"

    def call(self, func: Callable, *args, **kwargs):
        if self.state == "open":
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = "half-open"
            else:
                raise Exception("Circuit breaker is OPEN — fast failing")

        try:
            result = func(*args, **kwargs)
            if self.state == "half-open":
                self.state = "closed"
                self.failures = 0
            return result
        except Exception as e:
            self.failures += 1
            self.last_failure_time = time.time()
            if self.failures >= self.failure_threshold:
                self.state = "open"
            raise e


breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=2.0)

def flaky_service():
    if random.random() < 0.7:
        raise Exception("Service error")
    return "OK"

for i in range(10):
    try:
        result = breaker.call(flaky_service)
        print(f"Attempt {i+1}: OK (state: {breaker.state})")
    except Exception as e:
        print(f"Attempt {i+1}: {e} (state: {breaker.state})")
    time.sleep(0.1)

## 3. Fallback Chain

Try models in priority order. On failure, move to the next. A simple in-memory cache prevents redundant LLM calls for identical queries.

```
query → cache hit? → return cached
      → gpt-4o-mini → success? → return + cache
      → gpt-4o      → success? → return + cache
      → claude      → success? → return + cache
      → all failed  → raise
```

In [ ]:
class FallbackChain:
    """Try multiple models in order until one succeeds."""

    def __init__(self):
        self.models = [
            ("gpt-4o-mini", ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10)),
            ("gpt-4o",      ChatOpenAI(model="gpt-4o",      temperature=0, timeout=10)),
        ]
        self.cache = {}

    @traceable(name="fallback_invoke")
    def invoke(self, query: str, use_cache: bool = True) -> tuple[str, str]:
        """Returns (response, model_used). model_used='cache' on cache hit."""
        if use_cache and query in self.cache:
            return self.cache[query], "cache"

        errors = []
        for model_name, model in self.models:
            try:
                response = model.invoke(query)
                self.cache[query] = response.content
                return response.content, model_name
            except Exception as e:
                errors.append(f"{model_name}: {e}")

        raise Exception(f"All models failed: {errors}")


chain = FallbackChain()

for query in ["What is 2 + 2?", "What is Python?", "What is 2 + 2?"]:
    result, model = chain.invoke(query)
    print(f"[{model:12s}] {query} → {result[:50]}...")

## 4. Robust LangGraph Agent (Retry in Graph)

Error handling can be embedded directly in the graph via a routing function that checks `state["success"]`. If processing failed and retries remain, the conditional edge loops back to `process`. If retries are exhausted, it routes to `handle_error` instead.

```
START → process → [success] → finalize → END
              └── [retry]   → process (loop)
              └── [error]   → handle_error → END
```

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class RobustState(TypedDict):
    messages: Annotated[list, operator.add]
    error: Optional[str]
    retry_count: int
    max_retries: int
    success: bool


def create_robust_agent():
    def process_with_retry(state: RobustState) -> dict:
        try:
            # Simulate occasional failure on early retries
            if random.random() < 0.3 and state["retry_count"] < 2:
                raise Exception("Simulated processing error")
            response = llm.invoke(state["messages"])
            return {"messages": [response], "success": True, "error": None}
        except Exception as e:
            return {"error": str(e), "retry_count": state["retry_count"] + 1, "success": False}

    def should_continue(state: RobustState) -> Literal["retry", "error", "success"]:
        if state["success"]:
            return "success"
        elif state["retry_count"] < state["max_retries"]:
            return "retry"
        else:
            return "error"

    def handle_error(state: RobustState) -> dict:
        return {"messages": [AIMessage(content=f"Error after {state['max_retries']} retries: {state['error']}")]}

    def finalize(state: RobustState) -> dict:
        return state

    g = StateGraph(RobustState)
    g.add_node("process", process_with_retry)
    g.add_node("handle_error", handle_error)
    g.add_node("finalize", finalize)
    g.add_edge(START, "process")
    g.add_conditional_edges(
        "process", should_continue,
        {"retry": "process", "error": "handle_error", "success": "finalize"},
    )
    g.add_edge("handle_error", END)
    g.add_edge("finalize", END)
    return g.compile()


agent = create_robust_agent()

for i in range(3):
    result = agent.invoke({
        "messages": [HumanMessage(content="Hello!")],
        "error": None, "retry_count": 0, "max_retries": 3, "success": False,
    })
    status = "Success" if result["success"] else "Failed"
    print(f"Run {i+1}: {status} | Retries: {result['retry_count']} | {result['messages'][-1].content[:60]}...")

## 5. Structured Logging with JSONFormatter

`JSONFormatter` converts log records to JSON objects. This makes logs parseable by log aggregation systems (Datadog, CloudWatch, ELK). The `extra_data` dict is merged into the JSON at the top level — use it for request-specific metadata like user IDs, latency, or token counts.

In [ ]:
class JSONFormatter(logging.Formatter):
    def format(self, record):
        log_obj = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
            "module": record.module,
            "function": record.funcName,
        }
        if hasattr(record, "extra_data"):
            log_obj.update(record.extra_data)
        return json.dumps(log_obj)


def setup_logging():
    logger = logging.getLogger("langgraph_app")
    logger.setLevel(logging.INFO)
    if not logger.handlers:  # avoid duplicate handlers on re-run
        handler = logging.StreamHandler()
        handler.setFormatter(JSONFormatter())
        logger.addHandler(handler)
    return logger


logger = setup_logging()
logger.info("Application starting")
logger.info(
    "LLM request completed",
    extra={"extra_data": {"latency_ms": 245.3, "input_tokens": 80, "output_tokens": 120}},
)
logger.warning(
    "Rate limit approaching",
    extra={"extra_data": {"current_rate": 18, "limit": 20}},
)

## 6. MetricsCollector

`MetricsCollector.record_request()` accumulates raw counters. `get_summary()` derives derived metrics (error rate, average latency, cache hit rate) on demand. In production you'd push these to Prometheus, Datadog, or CloudWatch.

In [ ]:
class MetricsCollector:
    def __init__(self):
        self.metrics = {
            "requests_total": 0, "errors_total": 0,
            "latency_sum": 0,    "latency_count": 0,
            "tokens_input": 0,   "tokens_output": 0,
            "cache_hits": 0,     "cache_misses": 0,
        }

    def record_request(self, latency_ms: float, input_tokens: int = 0,
                       output_tokens: int = 0, error: bool = False, cache_hit: bool = False):
        self.metrics["requests_total"] += 1
        self.metrics["latency_sum"]    += latency_ms
        self.metrics["latency_count"]  += 1
        self.metrics["tokens_input"]   += input_tokens
        self.metrics["tokens_output"]  += output_tokens
        if error:     self.metrics["errors_total"] += 1
        if cache_hit: self.metrics["cache_hits"]   += 1
        else:         self.metrics["cache_misses"]  += 1

    def get_summary(self) -> dict:
        n = self.metrics["requests_total"]
        hits = self.metrics["cache_hits"]
        total_cache = hits + self.metrics["cache_misses"]
        return {
            "total_requests":    n,
            "error_rate":        f"{self.metrics['errors_total']/n:.2%}" if n else "0%",
            "avg_latency_ms":    round(self.metrics["latency_sum"] / self.metrics["latency_count"], 2) if n else 0,
            "total_tokens":      self.metrics["tokens_input"] + self.metrics["tokens_output"],
            "cache_hit_rate":    f"{hits/total_cache:.2%}" if total_cache else "0%",
        }


metrics = MetricsCollector()
metrics.record_request(latency_ms=245, input_tokens=80,  output_tokens=120, cache_hit=False)
metrics.record_request(latency_ms=5,   input_tokens=0,   output_tokens=0,   cache_hit=True)
metrics.record_request(latency_ms=890, input_tokens=200, output_tokens=350, error=True)

import pprint; pprint.pprint(metrics.get_summary())

## 7. Model Router — Route by Complexity

Use the cheap model (`gpt-4o-mini`) for simple queries and the expensive model (`gpt-4o`) only when complexity warrants it. The classifier itself runs on the cheap model — overhead is small but savings can be significant at scale.

In [ ]:
class ModelRouter:
    def __init__(self):
        self.cheap_model     = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.expensive_model = ChatOpenAI(model="gpt-4o",      temperature=0)
        self.classifier      = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    def classify_complexity(self, query: str) -> str:
        prompt = ChatPromptTemplate.from_template(
            "Classify this query as 'simple' or 'complex'.\n"
            "Simple: facts, short answers, arithmetic.\n"
            "Complex: analysis, reasoning, multi-step problems.\n\n"
            "Query: {query}\n\nRespond with only: simple or complex"
        )
        return self.classifier.invoke(prompt.format(query=query)).content.strip().lower()

    @traceable(name="routed_query")
    def invoke(self, query: str) -> tuple[str, str, float]:
        """Returns (response, model_used, estimated_cost)."""
        complexity = self.classify_complexity(query)
        if complexity == "simple":
            model, model_name, cost_per_1k = self.cheap_model,     "gpt-4o-mini", 0.00015
        else:
            model, model_name, cost_per_1k = self.expensive_model,  "gpt-4o",      0.0025

        response = model.invoke(query)
        tokens = len(query.split()) * 1.3
        return response.content, model_name, (tokens / 1000) * cost_per_1k


router = ModelRouter()
queries = [
    "What is 2 + 2?",
    "Analyze the economic implications of AI on the job market.",
    "What color is the sky?",
]

total_cost = 0
for q in queries:
    result, model, cost = router.invoke(q)
    total_cost += cost
    print(f"[{model:12s}] ${cost:.6f}  {q[:45]}...")
print(f"\nTotal estimated cost: ${total_cost:.6f}")

## 8. Semantic Cache + Token Budget

`CachedLLM` wraps any LLM and skips the API call on cache hit. Here the cache uses MD5 on the normalized (lowercased, trimmed) query — so `"What is Python?"` and `"what is python?"` hit the same cache entry.

`TokenBudget` rejects requests that exceed a per-request token ceiling, preventing runaway costs from pathologically long inputs.

In [ ]:
class SemanticCache:
    def __init__(self):
        self.cache = {}

    def _hash(self, query: str) -> str:
        return hashlib.md5(query.lower().strip().encode()).hexdigest()

    def get(self, query: str) -> Optional[str]:
        return self.cache.get(self._hash(query))

    def set(self, query: str, response: str):
        self.cache[self._hash(query)] = response


class CachedLLM:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.cache = SemanticCache()
        self.hits = 0
        self.misses = 0

    def invoke(self, query: str) -> tuple[str, bool]:
        cached = self.cache.get(query)
        if cached:
            self.hits += 1
            return cached, True
        self.misses += 1
        result = self.llm.invoke(query).content
        self.cache.set(query, result)
        return result, False


class TokenBudget:
    def __init__(self, max_tokens: int = 4000):
        self.max_tokens = max_tokens

    def estimate(self, text: str) -> int:
        return int(len(text.split()) * 1.3)

    def check(self, text: str):
        tokens = self.estimate(text)
        if tokens > self.max_tokens:
            raise ValueError(f"Query exceeds budget: {tokens} > {self.max_tokens} tokens")


# --- Caching demo ---
cached_llm = CachedLLM()
for query in ["What is Python?", "What is JavaScript?", "What is Python?", "what is python?"]:
    result, from_cache = cached_llm.invoke(query)
    source = "CACHE" if from_cache else "LLM  "
    print(f"[{source}] {query:30s} → {result[:40]}...")
print(f"\nHits: {cached_llm.hits}  Misses: {cached_llm.misses}  Rate: {cached_llm.hits/(cached_llm.hits+cached_llm.misses):.0%}")

print()

# --- Budget demo ---
budget = TokenBudget(max_tokens=100)
for q in ["What is AI?", "Explain " + "very " * 100 + "complex topic"]:
    try:
        budget.check(q)
        print(f"OK  : {q[:40]}...")
    except ValueError as e:
        print(f"OVER: {e}")